In [3]:
import arcpy
import numpy as np
import random
import math
import os
from datetime import datetime

# 设置工作空间
arcpy.env.workspace = r"D:\ArcGIS\sunleigangPro\模拟退火"
arcpy.env.overwriteOutput = True

# 输入栅格文件
input_raster = "最终ok.tif"

# 输出点要素类
output_points = "min_best_points.shp"
points = []

# 点的高度
point_height = 500  # 单位：米

# 视域范围（半径）
view_distance = 100  # 单位：米

# 目标可见区域比例
target_coverage = 0.9  # 90%
Imax=1000

# 将栅格数据加载到 NumPy 数组中
def raster_to_array(raster):
    raster_array = arcpy.RasterToNumPyArray(raster, nodata_to_value=0)
    return raster_array

# 计算栅格的有效面积
def calculate_raster_area(raster):
    cell_size = float(arcpy.GetRasterProperties_management(raster, "CELLSIZEX").getOutput(0))
    raster_array = raster_to_array(raster)
    valid_cell_count = np.count_nonzero(raster_array)
    raster_area = valid_cell_count * (cell_size ** 2)
    return raster_area

# 获取栅格的有效区域多边形
def get_raster_domain(raster):
    # 将栅格的有效区域转换为多边形
    domain_polygon = os.path.join(arcpy.env.workspace, "raster_domain.shp")
    arcpy.RasterDomain_3d(raster, domain_polygon, "POLYGON")
    return domain_polygon



# 计算视域覆盖率
def calculate_coverage():
    
    # 执行视域分析
    viewshed_result = arcpy.sa.Viewshed2(
        in_raster=input_raster,
        in_observer_features="min_temp_points.shp",
        out_agl_raster=None,
        analysis_type="FREQUENCY",
        vertical_error="0 Meters",
        out_observer_region_relationship_table=None,
        refractivity_coefficient=0.13,
        surface_offset="0 Meters",
        observer_elevation=point_height,
        observer_offset="1 Meters",
        inner_radius=None,
        inner_radius_is_3d="GROUND",
        outer_radius=view_distance,
        outer_radius_is_3d="GROUND",
        horizontal_start_angle=0,
        horizontal_end_angle=360,
        vertical_upper_angle=90,
        vertical_lower_angle=-90,
        analysis_method="ALL_SIGHTLINES",
        analysis_target_device="GPU_THEN_CPU"
    )
    
    # 将结果转换为 NumPy 数组
    viewshed_array = raster_to_array(viewshed_result)
    
    # 计算可见区域面积
    unique_values, counts = np.unique(viewshed_array, return_counts=True)
    cell_size = float(arcpy.GetRasterProperties_management(input_raster, "CELLSIZEX").getOutput(0))
    visible_area = 0
    for value, count in zip(unique_values, counts):
        if value > 0:  # 视域值大于0表示可见区域
            visible_area += count * (cell_size ** 2)
    
    # 计算覆盖率
    coverage = visible_area / raster_area
    
    shp_file = r"D:\ArcGIS\sunleigangPro\模拟退火\min_temp_points.shp"
    print(f"当前点数为：{arcpy.GetCount_management(shp_file)}，覆盖率为: {coverage}")
    return coverage

# 算法
def max_points(distance):
   
    print(f"当前时间为：******************{datetime.now()}*******************")
    # 删除旧的 temp_points.shp 文件
    max_temp_points = os.path.join(arcpy.env.workspace, "min_temp_points.shp")
    if arcpy.Exists(max_temp_points):
        arcpy.management.Delete(max_temp_points)
    i=1    
    best_coverage = 0
    
    # 循环执行
    while(i<Imax):
        random_seed = random.randint(1, 999999)  # 生成一个1到999999之间的随机整数
        random_generator = f"{random_seed} ACM599"  # 使用 ACM599 算法，但每次种子值不同
        with arcpy.EnvManager(randomGenerator=random_generator):
            arcpy.management.CreateRandomPoints(
                out_path=r"D:\ArcGIS\sunleigangPro\模拟退火",
                out_name="min_temp_points",
                constraining_feature_class="raster_domain",
                constraining_extent="DEFAULT",
                number_of_points_or_field=99999,
                minimum_allowed_distance=f"{distance} Meters",
                create_multipoint_output="POINT",
                multipoint_size=0
            )
        current_coverage = calculate_coverage()
        if current_coverage > best_coverage:
            best_coverage = current_coverage
            
        if best_coverage >= target_coverage:

            # 将 max_temp_points.shp 复制为 max_best_points.shp
            arcpy.management.CopyFeatures("min_temp_points.shp", "min_best_points.shp")
            break
        i += 1
    return best_coverage

# 动态调整点数以满足目标覆盖率
def optimize_coverage():
    
    distance = view_distance   #最小距离为视域半径
    best_coverage_flag = True
    
    while(best_coverage_flag):
        best_coverage = max_points(distance)
        if best_coverage >= target_coverage:
            best_feasible_coverage = best_coverage
            print(f"当前最优覆盖率 {best_coverage * 100}% 达到目标，减少点数增加距离：{distance+1}")
            distance += 1  
        else:
            print(f"当前最优覆盖率 {best_coverage * 100}% 未达到目标，最优距离为：{distance-1}")
            best_coverage_flag = False

    return  best_feasible_coverage

def extract_and_add_coordinates(output_points):
    # 读取生成的点文件，提取点坐标
    with arcpy.da.SearchCursor(output_points, ["SHAPE@XY"]) as cursor:
        for row in cursor:
            x, y = row[0]  # 提取点的 X 和 Y 坐标
            if not math.isnan(x) and not math.isnan(y):  # 检查坐标是否有效
                points.append((x, y))  # 将坐标添加到列表中
    
    # 添加 X 和 Y 坐标字段
    arcpy.management.AddField(output_points, "X", "DOUBLE")
    arcpy.management.AddField(output_points, "Y", "DOUBLE")
    
    # 更新点数据，将坐标写入 X 和 Y 字段
    with arcpy.da.UpdateCursor(output_points, ["SHAPE@XY", "X", "Y"]) as cursor:
        for i, row in enumerate(cursor):
            if i < len(points):  # 确保不超出 points 列表范围
                x, y = points[i]
                row[1] = x  # 更新 X 字段
                row[2] = y  # 更新 Y 字段
                cursor.updateRow(row)  # 提交更新
                        
# 主程序
if __name__ == "__main__":
    # 获取栅格的有效面积
    raster_area = calculate_raster_area(input_raster)
    print(f"栅格的有效面积: {raster_area} 平方米")
    
    # 获取栅格的有效区域多边形
    domain_polygon = get_raster_domain(input_raster)
    
    # 运行动态调整点数的优化算法
    best_feasible_coverage = optimize_coverage()
    
    extract_and_add_coordinates(output_points)
    
    # 输出结果
    print(f"最优点的数量: {len(points)}")
    print(f"最优点的位置: {points}")
    print(f"可见区域覆盖率: {best_feasible_coverage * 100}%")

    # 将生成的要素类加载到地图中
    aprx = arcpy.mp.ArcGISProject("CURRENT")
    map = aprx.listMaps()[0]  # 获取第一个地图
    map.addDataFromPath(os.path.join(arcpy.env.workspace, output_points))
    
    print("处理完成！")

栅格的有效面积: 572800.0 平方米
当前时间为：******************2025-02-20 09:57:04.954789*******************
当前点数为：43，覆盖率为: 0.9615921787709497
当前最优覆盖率 96.15921787709497% 达到目标，减少点数增加距离：101
当前时间为：******************2025-02-20 09:57:10.314301*******************
当前点数为：44，覆盖率为: 0.950768156424581
当前最优覆盖率 95.0768156424581% 达到目标，减少点数增加距离：102
当前时间为：******************2025-02-20 09:57:13.601537*******************
当前点数为：39，覆盖率为: 0.9429120111731844
当前最优覆盖率 94.29120111731844% 达到目标，减少点数增加距离：103
当前时间为：******************2025-02-20 09:57:19.186111*******************
当前点数为：39，覆盖率为: 0.9500698324022346
当前最优覆盖率 95.00698324022346% 达到目标，减少点数增加距离：104
当前时间为：******************2025-02-20 09:57:27.629369*******************
当前点数为：41，覆盖率为: 0.9587988826815642
当前最优覆盖率 95.87988826815642% 达到目标，减少点数增加距离：105
当前时间为：******************2025-02-20 09:57:35.377600*******************
当前点数为：38，覆盖率为: 0.9355796089385475
当前最优覆盖率 93.55796089385476% 达到目标，减少点数增加距离：106
当前时间为：******************2025-02-20 09:57:41.896608*******************
当前点数为：37，覆盖率为: 0

当前点数为：28，覆盖率为: 0.8362430167597765
当前点数为：29，覆盖率为: 0.852304469273743
当前点数为：30，覆盖率为: 0.8896648044692738
当前点数为：25，覆盖率为: 0.8093575418994413
当前点数为：28，覆盖率为: 0.864699720670391
当前点数为：29，覆盖率为: 0.8502094972067039
当前点数为：30，覆盖率为: 0.8517807262569832
当前点数为：26，覆盖率为: 0.8107541899441341
当前点数为：27，覆盖率为: 0.8252444134078212
当前点数为：29，覆盖率为: 0.8615572625698324
当前点数为：26，覆盖率为: 0.8145949720670391
当前点数为：28，覆盖率为: 0.825768156424581
当前点数为：27，覆盖率为: 0.833449720670391
当前点数为：29，覆盖率为: 0.8549231843575419
当前点数为：25，覆盖率为: 0.7978351955307262
当前点数为：28，覆盖率为: 0.8743016759776536
当前点数为：27，覆盖率为: 0.8035963687150838
当前点数为：28，覆盖率为: 0.8605097765363129
当前点数为：27，覆盖率为: 0.8329259776536313
当前点数为：27，覆盖率为: 0.8510824022346368
当前点数为：29，覆盖率为: 0.8793645251396648
当前点数为：30，覆盖率为: 0.9207402234636871
当前最优覆盖率 92.07402234636871% 达到目标，减少点数增加距离：125
当前时间为：******************2025-02-20 10:18:39.306224*******************
当前点数为：28，覆盖率为: 0.8685405027932961
当前点数为：29，覆盖率为: 0.8798882681564246
当前点数为：26，覆盖率为: 0.8222765363128491
当前点数为：28，覆盖率为: 0.8420041899441341
当前点数为

当前点数为：29，覆盖率为: 0.8716829608938548
当前点数为：25，覆盖率为: 0.8324022346368715
当前点数为：30，覆盖率为: 0.8482891061452514
当前点数为：26，覆盖率为: 0.8144203910614525
当前点数为：28，覆盖率为: 0.8399092178770949
当前点数为：26，覆盖率为: 0.8088337988826816
当前点数为：24，覆盖率为: 0.7863128491620112
当前点数为：26，覆盖率为: 0.8023743016759777
当前点数为：24，覆盖率为: 0.7932960893854749
当前点数为：29，覆盖率为: 0.866445530726257
当前点数为：25，覆盖率为: 0.7512220670391061
当前点数为：24，覆盖率为: 0.8128491620111732
当前点数为：28，覆盖率为: 0.8482891061452514
当前点数为：26，覆盖率为: 0.8311801675977654
当前点数为：27，覆盖率为: 0.8203561452513967
当前点数为：28，覆盖率为: 0.821054469273743
当前点数为：27，覆盖率为: 0.8324022346368715
当前点数为：26，覆盖率为: 0.8221019553072626
当前点数为：27，覆盖率为: 0.8475907821229051
当前点数为：26，覆盖率为: 0.8320530726256983
当前点数为：27，覆盖率为: 0.8461941340782123
当前点数为：26，覆盖率为: 0.7760125698324022
当前点数为：30，覆盖率为: 0.8612081005586593
当前点数为：23，覆盖率为: 0.767981843575419
当前点数为：28，覆盖率为: 0.8765712290502793
当前点数为：25，覆盖率为: 0.8067388268156425
当前点数为：26，覆盖率为: 0.8020251396648045
当前点数为：28，覆盖率为: 0.8285614525139665
当前点数为：25，覆盖率为: 0.7882332402234636
当前点数为：26，覆盖率为: 0.

当前点数为：28，覆盖率为: 0.8317039106145251
当前点数为：26，覆盖率为: 0.8207053072625698
当前点数为：25，覆盖率为: 0.8221019553072626
当前点数为：28，覆盖率为: 0.8418296089385475
当前点数为：28，覆盖率为: 0.8482891061452514
当前点数为：27，覆盖率为: 0.8056913407821229
当前点数为：25，覆盖率为: 0.7796787709497207
当前点数为：27，覆盖率为: 0.7908519553072626
当前点数为：27，覆盖率为: 0.8440991620111732
当前点数为：27，覆盖率为: 0.8203561452513967
当前点数为：24，覆盖率为: 0.754713687150838
当前点数为：26，覆盖率为: 0.8589385474860335
当前点数为：24，覆盖率为: 0.7730446927374302
当前点数为：25，覆盖率为: 0.8250698324022346
当前点数为：26，覆盖率为: 0.8481145251396648
当前点数为：26，覆盖率为: 0.8158170391061452
当前点数为：27，覆盖率为: 0.8393854748603352
当前点数为：27，覆盖率为: 0.8428770949720671
当前点数为：26，覆盖率为: 0.803072625698324
当前点数为：27，覆盖率为: 0.8341480446927374
当前点数为：27，覆盖率为: 0.8341480446927374
当前点数为：26，覆盖率为: 0.8034217877094972
当前点数为：27，覆盖率为: 0.8626047486033519
当前点数为：26，覆盖率为: 0.809532122905028
当前点数为：25，覆盖率为: 0.8056913407821229
当前点数为：29，覆盖率为: 0.8666201117318436
当前点数为：27，覆盖率为: 0.8568435754189944
当前点数为：26，覆盖率为: 0.8090083798882681
当前点数为：28，覆盖率为: 0.85143156424581
当前点数为：27，覆盖率为: 0.85

当前点数为：23，覆盖率为: 0.7451117318435754
当前点数为：28，覆盖率为: 0.8486382681564246
当前点数为：25，覆盖率为: 0.8280377094972067
当前点数为：27，覆盖率为: 0.8491620111731844
当前点数为：24，覆盖率为: 0.7932960893854749
当前点数为：27，覆盖率为: 0.840782122905028
当前点数为：29，覆盖率为: 0.8610335195530726
当前点数为：26，覆盖率为: 0.807786312849162
当前点数为：28，覆盖率为: 0.8564944134078212
当前点数为：29，覆盖率为: 0.8434008379888268
当前点数为：32，覆盖率为: 0.9059008379888268
当前最优覆盖率 90.59008379888269% 达到目标，减少点数增加距离：128
当前时间为：******************2025-02-20 11:44:03.684683*******************
当前点数为：27，覆盖率为: 0.8453212290502793
当前点数为：26，覆盖率为: 0.8337988826815642
当前点数为：26，覆盖率为: 0.8055167597765364
当前点数为：27，覆盖率为: 0.8454958100558659
当前点数为：27，覆盖率为: 0.8184357541899442
当前点数为：24，覆盖率为: 0.7592527932960894
当前点数为：25，覆盖率为: 0.7815991620111732
当前点数为：23，覆盖率为: 0.7648393854748603
当前点数为：25，覆盖率为: 0.78125
当前点数为：28，覆盖率为: 0.8252444134078212
当前点数为：27，覆盖率为: 0.8118016759776536
当前点数为：24，覆盖率为: 0.7852653631284916
当前点数为：25，覆盖率为: 0.7932960893854749
当前点数为：29，覆盖率为: 0.8543994413407822
当前点数为：28，覆盖率为: 0.8327513966480447
当前点数为：26，覆盖率为:

当前点数为：29，覆盖率为: 0.8505586592178771
当前点数为：27，覆盖率为: 0.833449720670391
当前点数为：26，覆盖率为: 0.7990572625698324
当前点数为：27，覆盖率为: 0.8283868715083799
当前点数为：29，覆盖率为: 0.8322276536312849
当前点数为：27，覆盖率为: 0.8502094972067039
当前点数为：26，覆盖率为: 0.8125
当前点数为：27，覆盖率为: 0.8259427374301676
当前点数为：24，覆盖率为: 0.745286312849162
当前点数为：27，覆盖率为: 0.8329259776536313
当前点数为：27，覆盖率为: 0.8430516759776536
当前点数为：24，覆盖率为: 0.7946927374301676
当前点数为：26，覆盖率为: 0.7896298882681564
当前点数为：27，覆盖率为: 0.8348463687150838
当前点数为：27，覆盖率为: 0.8191340782122905
当前点数为：28，覆盖率为: 0.8633030726256983
当前点数为：25，覆盖率为: 0.8386871508379888
当前点数为：26，覆盖率为: 0.7788058659217877
当前点数为：30，覆盖率为: 0.8552723463687151
当前点数为：29，覆盖率为: 0.8397346368715084
当前点数为：26，覆盖率为: 0.8245460893854749
当前点数为：29，覆盖率为: 0.8615572625698324
当前点数为：26，覆盖率为: 0.7932960893854749
当前点数为：28，覆盖率为: 0.8736033519553073
当前点数为：28，覆盖率为: 0.8386871508379888
当前点数为：28，覆盖率为: 0.8273393854748603
当前点数为：28，覆盖率为: 0.8371159217877095
当前点数为：26，覆盖率为: 0.8311801675977654
当前点数为：26，覆盖率为: 0.8100558659217877
当前点数为：26，覆盖率为: 0.81215083798

当前点数为：27，覆盖率为: 0.8481145251396648
当前点数为：28，覆盖率为: 0.8468924581005587
当前点数为：26，覆盖率为: 0.8310055865921788
当前点数为：26，覆盖率为: 0.8413058659217877
当前点数为：26，覆盖率为: 0.84375
当前点数为：26，覆盖率为: 0.8065642458100558
当前点数为：24，覆盖率为: 0.7758379888268156
当前点数为：27，覆盖率为: 0.8182611731843575
当前点数为：26，覆盖率为: 0.827513966480447
当前点数为：29，覆盖率为: 0.8564944134078212
当前点数为：28，覆盖率为: 0.8653980446927374
当前点数为：27，覆盖率为: 0.8563198324022346
当前点数为：27，覆盖率为: 0.8568435754189944
当前点数为：25，覆盖率为: 0.7931215083798883
当前点数为：25，覆盖率为: 0.8189594972067039
当前点数为：27，覆盖率为: 0.8229748603351955
当前点数为：25，覆盖率为: 0.8193086592178771
当前点数为：28，覆盖率为: 0.8353701117318436
当前点数为：29，覆盖率为: 0.8577164804469274
当前点数为：24，覆盖率为: 0.7784567039106145
当前点数为：28，覆盖率为: 0.8317039106145251
当前点数为：26，覆盖率为: 0.8008030726256983
当前点数为：28，覆盖率为: 0.869413407821229
当前点数为：25，覆盖率为: 0.7934706703910615
当前点数为：26，覆盖率为: 0.802199720670391
当前点数为：26，覆盖率为: 0.7831703910614525
当前点数为：26，覆盖率为: 0.8189594972067039
当前点数为：27，覆盖率为: 0.8701117318435754
当前点数为：26，覆盖率为: 0.8339734636871509
当前点数为：22，覆盖率为: 0.75628491620

当前点数为：27，覆盖率为: 0.8276885474860335
当前点数为：26，覆盖率为: 0.8170391061452514
当前点数为：26，覆盖率为: 0.8001047486033519
当前点数为：26，覆盖率为: 0.7761871508379888
当前点数为：25，覆盖率为: 0.8137220670391061
当前点数为：26，覆盖率为: 0.8128491620111732
当前点数为：26，覆盖率为: 0.7980097765363129
当前点数为：26，覆盖率为: 0.825768156424581
当前点数为：26，覆盖率为: 0.8290851955307262
当前点数为：27，覆盖率为: 0.8425279329608939
当前点数为：26，覆盖率为: 0.8051675977653632
当前点数为：26，覆盖率为: 0.8137220670391061
当前点数为：27，覆盖率为: 0.8362430167597765
当前点数为：25，覆盖率为: 0.8285614525139665
当前点数为：26，覆盖率为: 0.784217877094972
当前点数为：25，覆盖率为: 0.784217877094972
当前点数为：26，覆盖率为: 0.7934706703910615
当前点数为：26，覆盖率为: 0.822800279329609
当前点数为：28，覆盖率为: 0.8542248603351955
当前点数为：27，覆盖率为: 0.8449720670391061
当前点数为：27，覆盖率为: 0.8116270949720671
当前点数为：29，覆盖率为: 0.8650488826815642
当前点数为：27，覆盖率为: 0.8346717877094972
当前点数为：29，覆盖率为: 0.8440991620111732
当前点数为：28，覆盖率为: 0.8615572625698324
当前点数为：27，覆盖率为: 0.8308310055865922
当前点数为：25，覆盖率为: 0.8001047486033519
当前点数为：27，覆盖率为: 0.8470670391061452
当前点数为：26，覆盖率为: 0.8128491620111732
当前点数为：27，覆盖率为: 0.8

当前点数为：23，覆盖率为: 0.7807262569832403
当前点数为：25，覆盖率为: 0.8025488826815642
当前点数为：27，覆盖率为: 0.8297835195530726
当前点数为：26，覆盖率为: 0.8035963687150838
当前点数为：27，覆盖率为: 0.8207053072625698
当前点数为：29，覆盖率为: 0.8619064245810056
当前点数为：26，覆盖率为: 0.8301326815642458
当前点数为：25，覆盖率为: 0.8093575418994413
当前点数为：29，覆盖率为: 0.8393854748603352
当前点数为：27，覆盖率为: 0.8074371508379888
当前点数为：28，覆盖率为: 0.8598114525139665
当前点数为：24，覆盖率为: 0.7711243016759777
当前点数为：28，覆盖率为: 0.8519553072625698
当前点数为：26，覆盖率为: 0.833449720670391
当前点数为：27，覆盖率为: 0.8221019553072626
当前点数为：25，覆盖率为: 0.7805516759776536
当前点数为：27，覆盖率为: 0.8290851955307262
当前点数为：26，覆盖率为: 0.8198324022346368
当前点数为：27，覆盖率为: 0.8564944134078212
当前点数为：22，覆盖率为: 0.7442388268156425
当前点数为：26，覆盖率为: 0.8097067039106145
当前点数为：28，覆盖率为: 0.8358938547486033
当前点数为：25，覆盖率为: 0.8063896648044693
当前点数为：26，覆盖率为: 0.8051675977653632
当前点数为：26，覆盖率为: 0.8166899441340782
当前点数为：28，覆盖率为: 0.8062150837988827
当前点数为：24，覆盖率为: 0.7740921787709497
当前点数为：28，覆盖率为: 0.8585893854748603
当前点数为：22，覆盖率为: 0.6861033519553073
当前点数为：25，覆盖率为: 